In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import plotly.express as px


In [2]:
df = pd.read_csv("../lab data/amz_uk_price_prediction_dataset.csv")
df.head()

,uid,asin,title,stars,reviews,price,isBestSeller,boughtInLastMonth,category
0,1,B09B96TG33,"Echo Dot (5th generation, 2022 release) | Big ...",4.7,15308,21.99,False,0,Hi-Fi Speakers
1,2,B01HTH3C8S,"Anker Soundcore mini, Super-Portable Bluetooth...",4.7,98099,23.99,True,0,Hi-Fi Speakers
2,3,B09B8YWXDF,"Echo Dot (5th generation, 2022 release) | Big ...",4.7,15308,21.99,False,0,Hi-Fi Speakers
3,4,B09B8T5VGV,"Echo Dot with clock (5th generation, 2022 rele...",4.7,7205,31.99,False,0,Hi-Fi Speakers
4,5,B09WX6QD65,Introducing Echo Pop | Full sound compact Wi-F...,4.6,1881,17.99,False,0,Hi-Fi Speakers


## Part 1: Understanding Product Categories
Business Question: What are the most popular product categories on Amazon UK, and how do they compare in terms of listing frequency?

Frequency Tables:

Generate a frequency table for the product category.
Which are the top 5 most listed product categories?

Visualizations:

Display the distribution of products across different categories using a bar chart. If you face problems understanding the chart, do it for a subset of top categories.
For a subset of top categories, visualize their proportions using a pie chart. Does any category dominate the listings?

In [3]:

# to aggregate sales by category
category_sales = (
    df.groupby("category", as_index=False)["boughtInLastMonth"]
      .sum()
      .sort_values("boughtInLastMonth", ascending=False)
      .head(5)
)

category_sales


,category,boughtInLastMonth
126,Health & Personal Care,9626950
109,Grocery,8442550
20,Beauty,7052400
221,Pet Supplies,3231650
9,Baby,2635150


In [4]:

fig = px.bar(
    category_sales,
    x="boughtInLastMonth",
    y="category",
    orientation="h",
    title="Top 5 Categories by Sales"
)

# to make highest value appear at the top
fig.update_layout(yaxis=dict(autorange="reversed"))

fig.show()


In [ ]:
fig = px.histogram(df, x="boughtInLastMonth", y="category", title="Category sales")
fig.show()

In [ ]:
# Count products per category
category_counts = df["category"].value_counts()

# Convert to DataFrame
category_counts = category_counts.reset_index()
category_counts.columns = ["category", "count"]


fig = px.bar(
    category_counts,
    x="count",
    y="category",
    title="Number of Products per Category",
    orientation="h"
)

fig.show()


In [7]:
top_categories = category_counts.head(5)

fig = px.bar(
    top_categories,
    x="count",
    y="category",
    title="Top 5 Categories by Number of Products",
    orientation="h"
)

fig.show()


In [8]:
fig = px.pie(
    top_categories,
    names="category",
    values="count",
    title="Proportion of Products in Top 5 Categories"
)

fig.show()


## Part 2: Delving into Product Pricing
Business Question: How are products priced on Amazon UK, and are there specific price points or ranges that are more common?

Measures of Centrality:

Calculate the mean, median, and mode for the price of products.
What's the average price point of products listed? How does this compare with the most common price point (mode)?

Measures of Dispersion:

Determine the variance, standard deviation, range, and interquartile range for product price.
How varied are the product prices? Are there any indicators of a significant spread in prices?

Visualizations:

Is there a specific price range where most products fall? Plot a histogram to visualize the distribution of product prices. If its hard to read these diagrams, think why this is, and explain how it could be solved..
Are there products that are priced significantly higher than the rest? Use a box plot to showcase the spread and potential outliers in product pricing.

In [3]:
df.describe().round(2)

,uid,stars,reviews,price,boughtInLastMonth
count,2443651.00,2443651.00,2443651.00,2443651.00,2443651.00
mean,1370545.35,2.15,480.63,89.24,25.90
std,816056.17,2.19,5944.12,345.61,240.48
min,1.00,0.00,0.00,0.00,0.00
25%,658921.50,0.00,0.00,9.99,0.00
50%,1349029.00,0.00,0.00,19.09,0.00
75%,2060685.50,4.40,59.00,45.99,0.00
max,2828593.00,5.00,1356658.00,100000.00,50000.00


In [4]:
# Mean (average price)
mean_price = df["price"].mean().round(2)

# Median (middle price)
median_price = df["price"].median()

# Mode (most common price)
mode_price = df["price"].mode()[0]

print("Mean : ", mean_price)
print("Median : ", median_price)
print("Mode : ", mode_price)


Mean :  89.24
Median :  19.09
Mode :  9.99


The mean product price is 89.24, while the median price is considerably lower at 19.09 and the most common price point (mode) is 9.99. This large gap between the mean, median, and mode indicates a strongly right-skewed price distribution. The high mean relative to the median suggests the presence of high-priced outliers, which significantly inflate the average price, even though most products are priced at the lower end.

In [5]:
# Variance
price_variance = round(df["price"].var(),2)

# Standard deviation
price_std = round(df["price"].std(),2)

# Range
price_range = df["price"].max() - df["price"].min()

# Interquartile Range (IQR)
Q1 = df["price"].quantile(0.25)
Q3 = df["price"].quantile(0.75)
price_iqr = Q3 - Q1

print("Variance : ", price_variance)
print("Standart deviation : ", price_std)
print("Range : ", price_range)
print("Interquartile Range : ",price_iqr)


Variance :  119445.49
Standart deviation :  345.61
Range :  100000.0
Interquartile Range :  36.0


The product prices show significant variability. Although the interquartile range is small, indicating that most prices are concentrated within a narrow band, the large range and high standard deviation reveal the presence of extreme values. These outliers contribute to a substantial overall spread in prices.

In [ ]:
fig = px.histogram(
    df,
    x="price",
       title="Distribution of Product Prices"
)

fig.show()

The histogram is difficult to read because a small number of extremely high-priced products stretch the x-axis, compressing the majority of prices into a narrow region near zero. This right-skewed distribution obscures the underlying pattern of most product prices. The issue can be addressed by limiting the price range or complementing the histogram with a box plot.

In [ ]:
px.histogram(
    df[df["price"] <= 200],
    x="price",
    title="Distribution of Product Prices (≤ 200)"
).show()


In [ ]:
fig = px.box(
    df,
    x="price",
    title="Box Plot of Product Prices"
)

fig.show()

The horizontal box plot shows that most product prices are concentrated within a narrow, low price range, as indicated by the compact interquartile range near the lower end of the scale. However, several points appear far to the right of the upper whisker, revealing the presence of extreme high-priced outliers (77k and 100k). These outliers significantly increase the overall spread of prices and explain the large range and high standard deviation observed in the data.

In [6]:
df_w_outliars = df[df['price'] < 25000]

In [ ]:
fig = px.box(
    df_w_outliars,
    x="price",
    title="Box Plot of Product Prices"
)

fig.show()


## Part 3: Unpacking Product Ratings
Business Question: How do customers rate products on Amazon UK, and are there any patterns or tendencies in the ratings?

Measures of Centrality:

Calculate the mean, median, and mode for the rating of products.
How do customers generally rate products? Is there a common trend?

Measures of Dispersion:

Determine the variance, standard deviation, and interquartile range for product rating.
Are the ratings consistent, or is there a wide variation in customer feedback?

Shape of the Distribution:

Calculate the skewness and kurtosis for the rating column.
Are the ratings normally distributed, or do they lean towards higher or lower values?
Visualizations:

Plot a histogram to visualize the distribution of product ratings. Is there a specific rating that is more common?

In [7]:
df.head()

,uid,asin,title,stars,reviews,price,isBestSeller,boughtInLastMonth,category
0,1,B09B96TG33,"Echo Dot (5th generation, 2022 release) | Big ...",4.7,15308,21.99,False,0,Hi-Fi Speakers
1,2,B01HTH3C8S,"Anker Soundcore mini, Super-Portable Bluetooth...",4.7,98099,23.99,True,0,Hi-Fi Speakers
2,3,B09B8YWXDF,"Echo Dot (5th generation, 2022 release) | Big ...",4.7,15308,21.99,False,0,Hi-Fi Speakers
3,4,B09B8T5VGV,"Echo Dot with clock (5th generation, 2022 rele...",4.7,7205,31.99,False,0,Hi-Fi Speakers
4,5,B09WX6QD65,Introducing Echo Pop | Full sound compact Wi-F...,4.6,1881,17.99,False,0,Hi-Fi Speakers


In [8]:
# Mean (average stars)
mean_stars = df["stars"].mean().round(2)

# Median (middle stars)
median_stars = df["stars"].median()

# Mode (most common star)
mode_stars = df["stars"].mode()[0]

print("Mean : ", mean_stars)
print("Median : ", median_stars)
print("Mode : ", mode_stars)


Mean :  2.15
Median :  0.0
Mode :  0.0


The mean product rating is 2.15, while both the median and mode are 0.0. This indicates that most products receive no rating or very low ratings, with only a small number of higher-rated products increasing the average. Overall, customer ratings tend to be low, suggesting limited engagement or dissatisfaction with many products.

In [9]:
# Variance
stars_variance = round(df["stars"].var(),2)

# Standard deviation
stars_std = round(df["stars"].std(),2)

# Range
stars_range = df["stars"].max() - df["stars"].min()

# Interquartile Range (IQR)
Q1 = df["stars"].quantile(0.25)
Q3 = df["stars"].quantile(0.75)
stars_iqr = Q3 - Q1

print("Variance : ", stars_variance)
print("Standart deviation : ", stars_std)
print("Range : ", stars_range)
print("Interquartile Range : ", stars_iqr)


Variance :  4.82
Standart deviation :  2.19
Range :  5.0
Interquartile Range :  4.4


The dispersion measures show that product ratings vary widely. The standard deviation and interquartile range are large relative to the rating scale, and the full range of ratings is observed. This indicates inconsistent customer feedback, with significant variation in how products are rated.

In [10]:
# Skewness
rating_skewness = df["stars"].skew()

# Kurtosis
rating_kurtosis = df["stars"].kurtosis()

print("Skewness:", round(rating_skewness, 2))
print("Kurtosis:", round(rating_kurtosis, 2))


Skewness: 0.08
Kurtosis: -1.93


The skewness value of 0.08 indicates that the distribution of product ratings is approximately symmetric, with no strong bias toward higher or lower ratings. The negative kurtosis of −1.93 suggests a flatter distribution compared to a normal distribution, indicating that ratings are spread broadly across the scale rather than concentrated around a single value. Overall, the ratings are not normally distributed but show a wide and balanced spread.

In [11]:
fig = px.histogram(
    df,
    x="stars",
    title="Distribution of Product Ratings"
)

fig.show()


The histogram shows that a rating of 0 is the most common, indicating that many products are unrated or receive very low ratings. In contrast, products that do receive ratings are more commonly rated highly, particularly between 4 and 5 stars. This suggests a split pattern where many products have no ratings, while those that are rated tend to receive positive feedback.